## 0 · IMPORTS



In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# IMPORTS — Default libraries and packages
# ════════════════════════════════════════════════════════════════════════════
import sys
import os
from pathlib import Path

project_root = Path.cwd() / "Intertidal_analysis"
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import geopandas as gpd
import contextily as ctx
import openeo

# Custom modules
from intertidal.geometry import GeometryProcessor
from intertidal.raster import RasterProcessor
from intertidal.openeo_client import OpenEOClient
from intertidal.scl_processor import SCLProcessor
from intertidal.mapper import IntertidalMapper
from intertidal.tide_analyzer import TideAnalyzer
from intertidal.overpass import get_overpass_times
from intertidal.tide_metrics import calcular_metricas_completas,evaluar_calidad_distribucion,imprimir_metricas_completas

# Compatibility adapters
from intertidal.notebook_compat import (
    CoordinateUtils,
    OpenEOManager,
    download_date_rgb,
    download_date_scl,
    tif_to_rgb,
    tif_to_scl,
    compute_scl_stats,
    load_scl_stack,
    build_reference_map,
    compute_transition_cloud_stats,
    plot_scl_map,
    plot_reference_map,
    plot_rgb_grid,
    plot_water_frequency,
    download_reference_map_openeo,
    load_reference_map_tif,
    evaluate_transition_cloud_coverage_openeo,
    quantify_reference_gain,
    compute_water_frequency_openeo,
    get_water_centroid,
    plot_intertidal_map
)

# Tide modeling
from intertidal.tidemodel import PyTMDTideModel, CopernicusTideModel



# ── Plots en PNG a DPI alto (nítidos sin pixelar; descargables como imagen) ──
%config InlineBackend.figure_formats = ['png']
plt.rcParams['figure.dpi'] = 150          # nitidez también si se exporta a PNG
plt.rcParams['savefig.dpi'] = 200



## 1 · SETTING UP

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Defining the AOI (area of interest) 
# ════════════════════════════════════════════════════════════════════════════

# Site name
site = "Villaviciosa"


# Coords (DMS Format Polygon: degrees°minutes'seconds"hemisphere)
aoi_dms = [   
    '43°30\'24.99"N 5°25\'32.06"W',
    '43°29\'51.78"N 5°25\'12.91"W',
    '43°30\'37.72"N 5°23\'16.00"W',
    '43°30\'55.20"N 5°22\'31.65"W',
    '43°31\'39.66"N 5°23\'34.03"W',
]


# Creating geometry object from the DMS coordinates
coords = CoordinateUtils()
polygon = coords.make_polygon(aoi_dms)
bbox = coords.bbox_from_polygon(polygon) # Bounding box of the AOI taking into account max and min lat/lon values of the polygon

# Reference system used in this pipeline
crs = "EPSG:4326"  # WGS84
crs_web_mercator = "EPSG:3857"  # Web Mercator: used for web mapping and visualization

# Pixel resolution
pixel_resolution = 10  # meters
    
# Visualization 
fig, ax = plt.subplots(figsize=(10, 8))
gdf_wm = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326").to_crs("EPSG:3857")
b = gdf_wm.total_bounds
buffer = 0.12 * max(b[2] - b[0], b[3] - b[1])  # 12% del lado mayor: auto-ajuste al AOI
ax.set_xlim(b[0] - buffer, b[2] + buffer)
ax.set_ylim(b[1] - buffer, b[3] + buffer)
ax.set_aspect("equal")
ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.Esri.WorldImagery)  # zoom automatico -> mejor resolucion
gdf_wm.boundary.plot(ax=ax, color="red", linewidth=2, label="AOI")
ax.legend()
ax.set_title(f"Area of Interest — {site}", fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()



In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Defining the time range for the analysis
# ════════════════════════════════════════════════════════════════════════════

# Start and end dates for the analysis
start_date = "2016-01-01"
end_date = "2025-12-31"
time_extent = [start_date, end_date]

# ════════════════════════════════════════════════════════════════════════════
# Defining selected bands and thresholding
# ════════════════════════════════════════════════════════════════════════════

# SCL classes considered bad for the reference map (clouds, shadows, etc.)
bad_scl_classes = [3, 8, 9, 10] 

# Umbral NDWI para detectar agua (agua = NDWI > umbral). El reference map,
# el water frequency, el intertidal y la batimetria usan NDWI a 10 m REAL
# (B03/B08). 0.0 = umbral clasico; en rias turbias conviene bajarlo. DEA usa 0.1.
ndwi_threshold = 0.0

# Umbralizatation used to select strictly scenes with high clarity (i.e., low cloud coverage) for the reference map
ref_bad_fraction_threshold = 0.05  # 5% of bad pixels in the scene

# Stability threshold for the reference map: fraction of clear observations required to consider a pixel as stable (i.e., not changing over time)
ref_stable_threshold = 0.95

# Spatial buffer surrounding the water/land interface to define the intertidal zone (in pixels)
ref_coastal_buffer_pixels = 10

# Cloud coverage threshold for the transition map (> ref_bad_fraction_threshold is required for recovery)
transition_cloud_threshold = 0.1

# Intertidal zone thresholds by water frequecy
# OJO NDWI: estos umbrales del water frequency estaban tuneados para la WF de
# SCL. Con la WF de NDWI la distribución puede cambiar -> revisar el intertidal
# resultante y reajustar low/high (y ndwi_threshold) si hace falta.
low_threshold = 0.05
high_threshold = 0.85  

# ════════════════════════════════════════════════════════════════════════════
# Tide modeling parameters
# ════════════════════════════════════════════════════════════════════════════

tide_location = None # Automatically sets to the centroid of the transition zone (containing water), other format is (lat,lon)
model_name = "GOT4.10" #Options: GOT4.10, GOT4.7, GOT4.7b, TPXO9-atlas, TPXO9-atlas-v2, TPXO9-atlas-v3, TPXO9-atlas-v4, TPXO9-atlas-v5, TPXO9-atlas-v6, TPXO9-atlas-v7, TPXO9-atlas-v8, TPXO9-atlas-v9, TPXO9-atlas-v10, TPXO9-atlas-v11, TPXO9-atlas-v12, TPXO9-atlas-v13, TPXO9-atlas-v14, TPXO9-atlas-v15, TPXO9-atlas-v16, TPXO9-atlas-v17, TPXO9-atlas-v18, TPXO9-atlas-v19, TPXO9-atlas-v20, FES2014, FES2014b, FES2014c, FES2014d, FES2014e, FES2014f, FES2014g, FES2014h, FES2014i, FES2014j, FES2014k, FES2014l, FES2014m, FES2014n, FES2014o, FES2014p, FES2014q, FES2014r, FES2014s, FES2014t, CMEMS

# Tidal point (centroid if not given)
if tide_location is None:
    gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
    centroid = gdf.geometry.iloc[0].centroid
    tide_location = (centroid.y, centroid.x)  # (lat, lon)

selected_period = ["2016-01-01", "2025-12-31"]  # Period for tide analysis
spread = 2 # In hours
# ════════════════════════════════════════════════════════════════════════════
# OUTPUTS — Defining output directories and file paths
# ════════════════════════════════════════════════════════════════════════════

# Output directory for the analysis results
output_dir = Path(".")
ref_map_path = "reference_map.tif"

# ════════════════════════════════════════════════════════════════════════════
# SUMMING UP THE PARAMETERS
# ════════════════════════════════════════════════════════════════════════════

print(f"Temporal range: {time_extent[0]} → {time_extent[1]}")
print(f"Clean scenes threshold (reference): ≤{ref_bad_fraction_threshold*100:.0f}% bad pixels")
print(f"Pixel stability threshold:           ≥{ref_stable_threshold*100:.0f}% coincident observations")
print(f"Coastal buffer:                      {ref_coastal_buffer_pixels} px ({ref_coastal_buffer_pixels*20} m)")
print(f"Transition cloud threshold : ≤{transition_cloud_threshold*100:.0f}%")

## 2 · LINKING UP WITH COPERNICUS DATASPACE


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Connecting to the OpenEO backend and initializing the client
# ════════════════════════════════════════════════════════════════════════════

# Using OpenEOManager class to maintain modularity, but you can also directly use the OpenEOClient class if you prefered
manager = OpenEOManager()
manager.connect()

# Extracting the connection in order to use it in the rest of the pipeline
conn = manager.connection

print(f"Connected to OpenEO backend. Account: {conn.describe_account()['name']}")

## 3 · MEASURING WHAT ARE WE WORKING WITH

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Exploring the available dates for the AOI and time range
# ════════════════════════════════════════════════════════════════════════════

# In order to quantify the amount of images we will process, we query Sentinel-2 L2A data via the STAC Catalog, which only reads metadata and does no computation
overpass_times = get_overpass_times(bbox, time_extent) #returns aquisition dates ('YYYY-MM-DD')
all_dates= sorted(overpass_times.keys())

# Displaying the result
print(f"Available dates in the time range {time_extent[0]} → {time_extent[1]}: {len(all_dates)} dates found")
for date in all_dates:
    print(date)


## 4 · BUILDING UP THE REFERENCE MAP

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Sección 4 · Reference map — DESCARGA ÚNICA (agua por NDWI a 10 m real)
# `analyze_ndwi_cube_openeo` baja B03/B08/SCL UNA sola vez y calcula en LOCAL el
# reference map, la cobertura de nubes (sección 5) y el water frequency (sección
# 6), en lugar de lanzar 3 jobs separados. El agua se detecta por NDWI a 10 m REAL (B03/B08); el SCL solo
# es máscara de nubes. Cálculo en streaming (RAM acotada, escala a km²).
# ════════════════════════════════════════════════════════════════════════════
from intertidal.notebook_compat import analyze_ndwi_cube_openeo

analysis = analyze_ndwi_cube_openeo(
    conn=conn,
    bbox=bbox,
    time_extent=time_extent,
    ndwi_threshold=ndwi_threshold,
    bad_classes=bad_scl_classes,
    bad_fraction_threshold=ref_bad_fraction_threshold,
    stable_threshold=ref_stable_threshold,
    transition_buffer_pixels=ref_coastal_buffer_pixels,
    global_bad_fraction_threshold=ref_bad_fraction_threshold,
    transition_cloud_threshold=transition_cloud_threshold,
    reference_dates=[],
    verbose=False, plot=False,        # el reporte se muestra por secciones (aquí y en la 5)
    # cache_nc="ndwi_cube_cache.nc",   # descomenta para REUSAR una descarga previa (mismo AOI)
)

# Variables del reference map (mismos nombres que antes)
reference_map_openeo   = analysis.reference_map
ref_transform, ref_crs = analysis.transform, analysis.crs
transition_mask        = (reference_map_openeo == 0)
analysis.save_reference_map(ref_map_path)        # guarda reference_map.tif

# Stats + visualización del reference map
analysis.report_reference_map()


## 5 · RECOVERING USABLE DATA

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Sección 5 · Recuperación de datos usables — nubes en la zona de transición
# Usa el MISMO cubo ya descargado en la sección 4 (no vuelve a bajar nada).
# ════════════════════════════════════════════════════════════════════════════
analysis.report_cloud_coverage()

# Variables aguas abajo (mismos nombres que antes)
transition_stats        = analysis.transition_stats
valid_dates_transition  = analysis.valid_dates
reference_dates         = analysis.reference_dates
newly_valid             = analysis.newly_valid
gain                    = analysis.gain


## 6 · DEALING WITH WATER FREQUENCY

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Sección 6 · Water frequency — desde el cubo ya descargado (sin nuevo job)
# ════════════════════════════════════════════════════════════════════════════
valid_dates_for_wf = sorted(valid_dates_transition)
print(f"Valid dates for water frequency computation: {len(valid_dates_for_wf)}")

if valid_dates_for_wf:
    print(f"  Range: {valid_dates_for_wf[0]} → {valid_dates_for_wf[-1]}")
else:
    raise ValueError("No valid dates available for water frequency computation.")

wf_out_out_path = "water_frequency.tif"

# Reusa el netCDF (B03/B08/SCL) de `analysis` en streaming (agua por NDWI),
# sin lanzar otro batch job
water_freq, wf_transform, wf_crs = analysis.water_frequency(
    valid_dates=valid_dates_for_wf,
    out_path=wf_out_out_path,
    min_obs=8,
)

print("Water frequency computed:")
print(f"  Shape: {water_freq.shape}")
print(f"  CRS:   {wf_crs}")
print(f"  Values: [{np.nanmin(water_freq):.3f}, {np.nanmax(water_freq):.3f}]")
print(f"  Pixels with data (non-NaN): {np.isfinite(water_freq).sum():,}")

plot_water_frequency(
    water_freq,
    transform=wf_transform,
    crs=wf_crs,
    polygon=polygon,
    title="Water frequency map - Ria de Villaviciosa",
    min_water_patch_pixels=20,
    water_presence_threshold=0.15,
)


## 7 · WHERE IS THE INTERTIDAL ZONE

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Determination of the intertidal zone based on the water frequency map and the reference map
# ════════════════════════════════════════════════════════════════════════════

# The 3 criteria for the intertidal zone are:
intertidal_mask = (
    (reference_map_openeo == 0) &  # Transition zone in the reference map
    (water_freq >= low_threshold) &  # Water frequency above the low threshold
    (water_freq <= high_threshold)  # Water frequency below the high threshold
)

# Counting the number of intertidal pixels
total_intertidal = intertidal_mask.sum()
area_km2 = total_intertidal * (pixel_resolution ** 2) / 1e6  

# Displaying the results
print(f"Intertidal zone determination:")
print(f"─────────────────────────────────────")
print(f"Transition zone (reference map): {(reference_map_openeo==0).sum():>8,} px")
print(f"  → intertidal ({low_threshold:.2f} ≤ WF ≤ {high_threshold:.2f}): {total_intertidal:>8,} px  ({area_km2:.2f} km²)")
print(f"  → terrestrial (WF < {low_threshold:.2f}):   {((reference_map_openeo==0) & (water_freq <= low_threshold)).sum():>8,} px")
print(f"  → aquatic (WF > {high_threshold:.2f}):              {((reference_map_openeo==0) & (water_freq >= high_threshold)).sum():>8,} px")

# Defining the output path for the intertidal zone map
intertidal_out_path = "intertidal_mask.tif"

# Visualizing the intertidal zone map
plot_intertidal_map(
    intertidal_mask = intertidal_mask,
    wf_transform=wf_transform,
    wf_crs=wf_crs,
    intertidal_out_path=intertidal_out_path,
    low_threshold=low_threshold,
    high_threshold=high_threshold,
    area_km2=area_km2,
    title="Intertidal map: Ria de Villaviciosa"
    )


## 8 · TIME FOR THE TIDES

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Obtention of the tide values for the intertidal zone based on the tide model and the centroid of the intertidal zone
# ════════════════════════════════════════════════════════════════════════════
from intertidal import notebook_compat

tide_model = PyTMDTideModel(
    model_name=model_name,
    directory=str(output_dir / "tide_models"),
    box_size = 2.0, # In degrees, the size of the box around the point of interest
    resolution = 0.05
)

# Computing tide points following the spread and the period given
dates_reference = pd.date_range(start=selected_period[0], end=selected_period[1], freq=f'{spread}h')
print(f"  {len(dates_reference)} measurements (every {spread}h)\n")

# Computing tide values for the point of interest
heights = tide_model.get_tide_heights_batch(lat=tide_location[0], lon=tide_location[1], datetimes=[d.to_pydatetime() for d in dates_reference])

# Obtaining the range
range = max(heights) - min(heights)
print(f"  Tide range: {range:.2f} m (min: {min(heights):.2f} m, max: {max(heights):.2f} m)\n")
print(f"Standard deviation: {np.std(heights):.2f} m\n")

# ════════════════════════════════════════════════════════════════════════════
# Visualization of the tide values over time
# ════════════════════════════════════════════════════════════════════════════

notebook_compat.plot_tide_time_series(dates_reference, heights, title=f"Tide time series — {model_name} at {tide_location[0]:.4f}°N, {tide_location[1]:.4f}°W")

# ════════════════════════════════════════════════════════════════════════════
# Computing the stats of the distribution
# ════════════════════════════════════════════════════════════════════════════

# Doing the same for the valid dates
valid_heights = tide_model.get_tide_heights_batch(lat=tide_location[0],lon=tide_location[1],datetimes=valid_dates_for_wf)
range_valid = max(valid_heights) - min(valid_heights)
print(f"  Tide range (valid dates): {range_valid:.2f} m (min: {min(valid_heights):.2f} m, max: {max(valid_heights):.2f} m)\n")
print(f"Standard deviation (valid dates): {np.std(valid_heights):.2f} m\n")

# Casting into numpy arrays for computational ease
reference_values = np.array(heights)
valid_values = np.array(valid_heights)

# Computing the complete stats and displaying them in a formatted output
metrics = calcular_metricas_completas(valid_values,reference_values)
imprimir_metricas_completas(metrics)

# ════════════════════════════════════════════════════════════════════════════
# Looking for a bias in the valid dates time series
# ════════════════════════════════════════════════════════════════════════════

notebook_compat.plot_tide_time_series(valid_dates_for_wf, valid_heights, title=f"Tide time series for only valid dates — {model_name} at {tide_location[0]:.4f}°N, {tide_location[1]:.4f}°W")
notebook_compat.plot_tide_distribution(heights,valid_heights,model_name,title="Vertical distribution of valid heights")


## 9 · DEM: HOW HIGH ARE WE?

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Bathymetry / DEM reconstruction
# ════════════════════════════════════════════════════════════════════════════

from intertidal.bathymetry import BathymetryReconstructor

# Relación fecha -> altura de marea
tide_dict = dict(zip(valid_dates_for_wf, valid_heights))

# Crear reconstruidor
reconstructor = BathymetryReconstructor(conn)

# Ejecutar reconstrucción
result = reconstructor.reconstruct(
    bbox=bbox,
    time_extent=time_extent,
    tide_heights=tide_dict,
    valid_dates=valid_dates_for_wf,
    out_dir="bathymetry",
    ndwi_threshold=ndwi_threshold,
    force=True,      # False para reutilizar resultados existentes
)

# Extraer productos
bathymetry = result.elevation
confidence = result.confidence
residual = result.residual
observation_count = result.observation_count
slope = result.slope
aspect = result.aspect
hillshade = result.hillshade
contours = result.contours

print("Bathymetry successfully reconstructed")
print(f"Shape: {bathymetry.shape}")
print(f"Elevation range: {np.nanmin(bathymetry):.2f} → {np.nanmax(bathymetry):.2f} m")
print(f"Mean confidence: {np.nanmean(confidence):.3f}")
print(f"Mean residual: {np.nanmean(residual):.3f} m")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Bathymetry visualization
# ════════════════════════════════════════════════════════════════════════════
import importlib

import intertidal.visualization
import intertidal.notebook_compat
import intertidal.bathymetry

importlib.reload(intertidal.visualization)
importlib.reload(intertidal.notebook_compat)
importlib.reload(intertidal.bathymetry)

# --------------------------------------------------------------------------
# Transecto (atravesando aproximadamente la ría)
# (row, col)
# --------------------------------------------------------------------------
start = (139, 310)   # flats Misiego (SW) - transecto de campo
end   = (95, 355)    # Observatorio de aves de Misiego (NE)

# --------------------------------------------------------------------------
# 1. Bathymetry
# --------------------------------------------------------------------------
notebook_compat.plot_bathymetry(
    elevation=bathymetry,
    confidence=confidence,
    hillshade=hillshade,
    contour_interval=0.25,
    min_confidence=0.0,  
)

# --------------------------------------------------------------------------
# 2. Confidence
# --------------------------------------------------------------------------

notebook_compat.plot_bathymetry_uncertainty(
    confidence=confidence,
    mask=intertidal_mask,   # la máscara de tu segunda imagen (WF ∈ [0.01, 0.99])
    elevation=bathymetry,
)
# --------------------------------------------------------------------------
# 3. Bathymetric transect
# --------------------------------------------------------------------------
notebook_compat.plot_bathymetry_profile(
    elevation=bathymetry,
    mask=intertidal_mask,
    start=start,
    end=end,
    pixel_size=10.0,
)
# --------------------------------------------------------------------------
# 4. Interactive 3D
# --------------------------------------------------------------------------

notebook_compat.plot_bathymetry_3d(
    elevation=bathymetry,
    mask=intertidal_mask,
    confidence=confidence,
    pixel_size=10,
    vertical_exaggeration=5,
    downsample=3,
)

## Comparación de métodos de batimetría

Dos reconstrucciones alternativas (locales) sobre las mismas fechas válidas, para comparar con las mismas 4 visualizaciones:

- **Método 1 — Diccionario de píxeles:** por píxel, sus observaciones agua/tierra y las mareas acotan la elevación `[zmin, zmax]`; se toma el punto medio.
- **Método 2 — Isolíneas (waterlines):** la línea de costa de cada fecha (z = marea) forma una nube de puntos con la que se interpola el DEM.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Método 1 — Diccionario de píxeles (bracketing por píxel)
# ════════════════════════════════════════════════════════════════════════════
import importlib
import intertidal.bathymetry, intertidal.visualization, intertidal.notebook_compat
importlib.reload(intertidal.bathymetry)
importlib.reload(intertidal.visualization)
importlib.reload(intertidal.notebook_compat)
from intertidal import notebook_compat
from intertidal.bathymetry import BathymetryReconstructor

reconstructor = BathymetryReconstructor(conn)

result_pixels = reconstructor.reconstruct_pixels(
    bbox=bbox,
    time_extent=time_extent,
    tide_heights=tide_dict,
    valid_dates=valid_dates_for_wf,
    out_dir="bathymetry",
    ndwi_threshold=ndwi_threshold,
    force=True,
)

print("Method 1 (pixel bracketing) reconstructed")
print(f"Shape: {result_pixels.elevation.shape}")
print(f"Reconstructed pixels: {int(result_pixels.mask.sum()):,} ({100*result_pixels.mask.mean():.2f}%)")
print(f"Elevation range: {np.nanmin(result_pixels.elevation):.2f} → {np.nanmax(result_pixels.elevation):.2f} m")
print(f"Mean confidence: {np.nanmean(result_pixels.confidence):.3f}")

In [ ]:
# Visualizaciones — Método 1 (diccionario de píxeles)
start = (139, 310)   # flats Misiego (SW) - transecto de campo
end   = (95, 355)    # Observatorio de aves de Misiego (NE)

notebook_compat.plot_bathymetry(
    elevation=result_pixels.elevation,
    confidence=result_pixels.confidence,
    hillshade=result_pixels.hillshade,
    contour_interval=0.25,
    title="Bathymetry — pixel dictionary",
)
notebook_compat.plot_bathymetry_uncertainty(
    confidence=result_pixels.confidence,
    mask=result_pixels.mask,
    elevation=result_pixels.elevation,
)
notebook_compat.plot_bathymetry_profile(
    elevation=result_pixels.elevation,
    mask=result_pixels.mask,
    start=start, end=end, pixel_size=10.0,
)
notebook_compat.plot_bathymetry_3d(
    elevation=result_pixels.elevation,
    mask=result_pixels.mask,
    confidence=result_pixels.confidence,
)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Método 2 — Isolíneas (waterlines)
# ════════════════════════════════════════════════════════════════════════════
result_isolines = reconstructor.reconstruct_isolines(
    bbox=bbox,
    time_extent=time_extent,
    tide_heights=tide_dict,
    valid_dates=valid_dates_for_wf,
    out_dir="bathymetry",
    ndwi_threshold=ndwi_threshold,
    force=True,
)

print("Method 2 (waterline isolines) reconstructed")
print(f"Shape: {result_isolines.elevation.shape}")
print(f"Reconstructed pixels: {int(result_isolines.mask.sum()):,} ({100*result_isolines.mask.mean():.2f}%)")
print(f"Elevation range: {np.nanmin(result_isolines.elevation):.2f} → {np.nanmax(result_isolines.elevation):.2f} m")
print(f"Mean confidence: {np.nanmean(result_isolines.confidence):.3f}")

In [ ]:
# Visualizaciones — Método 2 (isolíneas / waterlines)
notebook_compat.plot_bathymetry(
    elevation=result_isolines.elevation,
    confidence=result_isolines.confidence,
    hillshade=result_isolines.hillshade,
    contour_interval=0.25,
    title="Bathymetry — waterline isolines",
)
notebook_compat.plot_bathymetry_uncertainty(
    confidence=result_isolines.confidence,
    mask=result_isolines.mask,
    elevation=result_isolines.elevation,
)
notebook_compat.plot_bathymetry_profile(
    elevation=result_isolines.elevation,
    mask=result_isolines.mask,
    start=start, end=end, pixel_size=10.0,
)
notebook_compat.plot_bathymetry_3d(
    elevation=result_isolines.elevation,
    mask=result_isolines.mask,
    confidence=result_isolines.confidence,
)